# scVI Integration + Doublet Cluster Removal + Integration Metrics

Loads the shared preprocessed object (with HVGs already stored), runs scVI integration,
removes doublet-enriched clusters, and evaluates integration quality with scib-metrics
comparing **scVI vs unintegrated PCA**.

**Steps:**
1. Load preprocessed object
2. scVI integration (using saved HVGs) with early stopping
3. PCA baseline (unintegrated, for comparison)
4. UMAP + Leiden clustering
5. Doublet cluster identification + removal
6. scib benchmarking — scVI vs PCA
7. Save clean integrated object

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────
DATA_PATH  = "/vol/disk/ubuntu/master_practicum_cytokines/data/data_for_practicum_consensus_doublets_removed.h5ad"
OUTPUT_DIR = "/vol/disk/ubuntu/master_practicum_cytokines/lisa"

# ── library / sample column ────────────────────────────────────────────────
SAMPLE_COL = "library"

# ── clustering ─────────────────────────────────────────────────────────────
LEIDEN_RESOLUTION = 0.7

# ── doublet cluster removal ────────────────────────────────────────────────
DOUBLET_CLUSTER_THRESHOLD = 0.4

# ── derived ────────────────────────────────────────────────────────────────
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input:             {DATA_PATH}")
print(f"Output directory:  {OUTPUT_DIR}")
print(f"Leiden resolution: {LEIDEN_RESOLUTION}")

In [ ]:
%matplotlib inline

import gc
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
sc.settings.figdir = OUTPUT_DIR

def log_shape(adata, label):
    print(f"[{label}]  cells: {adata.n_obs:,}   genes: {adata.n_vars:,}")

## Step 1 — Load preprocessed object

In [ ]:
adata = sc.read_h5ad(DATA_PATH)
log_shape(adata, "Loaded")
print(f"\nHVGs available: {adata.var['hvg'].sum()}")
print(f"Doublet scores available: {'scDblFinder_score' in adata.obs.columns}")
adata

## Step 2 — scVI integration

In [ ]:
adata_scvi = adata[:, adata.var["hvg"]].copy()
print(f"scVI input: {adata_scvi.n_obs:,} cells × {adata_scvi.n_vars:,} HVGs")

scvi.model.SCVI.setup_anndata(adata_scvi, layer="counts", batch_key=SAMPLE_COL)
model = scvi.model.SCVI(adata_scvi, n_layers=2, n_latent=30, gene_likelihood="nb")

model.train(
    max_epochs=400,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_min_delta=0.0,
    plan_kwargs={"lr": 1e-3},
)

train_elbo = model.history["elbo_train"]
val_elbo   = model.history["elbo_validation"]
n_epochs   = len(train_elbo)

# ── Plot 1: train + validation ELBO ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_elbo.index, train_elbo["elbo_train"],   label="train ELBO")
ax.plot(val_elbo.index,   val_elbo["elbo_validation"], label="validation ELBO")
ax.axvline(n_epochs - 1, color="red", linestyle="--", label=f"stopped at epoch {n_epochs}")
ax.set_xlabel("Epoch"); ax.set_ylabel("ELBO")
ax.set_title("scVI training — ELBO curves")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "scvi_training_loss.png"), bbox_inches="tight")
plt.show()

# ── Plot 2: per-epoch improvement in validation ELBO ────────────────────────
val_series = val_elbo["elbo_validation"]
delta = val_series.diff().abs()

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=False)
ax = axes[0]
ax.plot(val_series.index[-200:], val_series.iloc[-200:], color="orange")
ax.set_ylabel("Validation ELBO"); ax.set_title("Last 200 epochs — still converging?")
ax = axes[1]
ax.plot(delta.index, delta, color="steelblue", linewidth=0.8, label="|Δ val ELBO|")
ax.set_yscale("log")
for md, color in [(0.1, "green"), (0.5, "orange"), (1.0, "red"), (2.0, "purple")]:
    ax.axhline(md, linestyle="--", color=color, linewidth=1, label=f"min_delta={md}")
ax.set_xlabel("Epoch"); ax.set_ylabel("|Δ ELBO| (log scale)")
ax.set_title("Per-epoch improvement — helps choose min_delta")
ax.legend(fontsize=8); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "scvi_early_stopping_analysis.png"), bbox_inches="tight")
plt.show()

# ── Overfitting check ────────────────────────────────────────────────────────
# Overfitting = validation ELBO increases while train ELBO keeps decreasing
# Check last 50 epochs: if val goes up but train goes down, that's a problem
last_n = 50
train_trend = train_elbo["elbo_train"].iloc[-last_n:].diff().mean()
val_trend   = val_elbo["elbo_validation"].iloc[-last_n:].diff().mean()
gap_start   = abs(val_elbo["elbo_validation"].iloc[10] - train_elbo["elbo_train"].iloc[10])
gap_end     = abs(val_elbo["elbo_validation"].iloc[-1]  - train_elbo["elbo_train"].iloc[-1])

print("\n── Overfitting check ──")
print(f"Train ELBO trend (last {last_n} epochs): {train_trend:+.4f} per epoch")
print(f"Val   ELBO trend (last {last_n} epochs): {val_trend:+.4f} per epoch")
print(f"Train/val gap at epoch 10:  {gap_start:.3f}")
print(f"Train/val gap at final epoch: {gap_end:.3f}")

if val_trend > 0 and train_trend < 0:
    print("⚠ WARNING: possible overfitting — val ELBO increasing while train decreasing")
elif gap_end > gap_start * 2:
    print("⚠ WARNING: train/val gap widened significantly — check ELBO plot")
else:
    print("✓ No overfitting detected — train and val ELBO converged together")

# ── Simulate early stopping ───────────────────────────────────────────────────
print("\nSimulated early stopping (patience=20):")
print(f"{'min_delta':>10}  {'stop_epoch':>10}  {'epochs_saved':>12}")
print("-" * 36)
for md in [0.1, 0.5, 1.0, 2.0]:
    patience_counter = 0
    stop_ep = n_epochs
    for ep in range(1, len(delta)):
        if delta.iloc[ep] < md:
            patience_counter += 1
            if patience_counter >= 20:
                stop_ep = ep
                break
        else:
            patience_counter = 0
    print(f"{md:>10.1f}  {stop_ep:>10}  {n_epochs - stop_ep:>12}")
print(f"\nActual epochs trained: {n_epochs} / 400")

adata.obsm["X_scVI"] = model.get_latent_representation()
del adata_scvi
gc.collect()
print("\nscVI latent representation stored in adata.obsm['X_scVI']")

## Step 3 — UMAP + Leiden clustering on scVI latent

## Step 3 — PCA baseline (unintegrated, for scib comparison)

In [ ]:
# Compute PCA on log-normalised HVGs — this is the unintegrated baseline
# scib will compare X_pca (no batch correction) vs X_scVI (batch-corrected)
adata.X = adata.layers["log1p_norm"]
adata.var["highly_variable"] = adata.var["hvg"]
sc.pp.pca(adata, svd_solver="arpack", mask_var="highly_variable")
print(f"PCA computed: {adata.obsm['X_pca'].shape[1]} components")
print("X_pca stored — unintegrated baseline for scib benchmarking")

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scVI", key_added="neighbors_scVI")
sc.tl.umap(adata, neighbors_key="neighbors_scVI")
adata.obsm["X_umap_scVI"] = adata.obsm["X_umap"].copy()

sc.tl.leiden(
    adata, neighbors_key="neighbors_scVI",
    key_added="leiden_scVI", resolution=LEIDEN_RESOLUTION,
    flavor="igraph", n_iterations=2, directed=False,
)
print(f"Leiden clusters (resolution={LEIDEN_RESOLUTION}): {adata.obs['leiden_scVI'].nunique()}")

# Leiden clusters + doublet score side by side
sc.pl.embedding(
    adata, basis="X_umap_scVI",
    color=["leiden_scVI", "scDblFinder_score"],
    vmax=[None, 1], vmin=[None, 0],
    ncols=2, legend_loc="on data",
    save="_scvi_clusters_vs_doublet_score.png",
)

# All QC metrics
qc_metrics = ["total_counts", "n_genes_by_counts", "pct_counts_mt",
               "pct_counts_ribo", "pct_counts_hb", "scDblFinder_score",
               "scDblFinder_class", SAMPLE_COL]
qc_metrics = [m for m in qc_metrics if m in adata.obs.columns]
sc.pl.embedding(
    adata, basis="X_umap_scVI",
    color=qc_metrics, ncols=3,
    save="_scvi_umap_qc.png",
)

## Step 4b — Differentially expressed genes between clusters

In [ ]:
# Check: confirm RPL/RPS/MT genes are absent from HVGs
hvg_genes   = adata.var_names[adata.var["hvg"]]
ribo_in_hvg = hvg_genes[hvg_genes.str.startswith("RPL") | hvg_genes.str.startswith("RPS")]
mt_in_hvg   = hvg_genes[hvg_genes.str.startswith("MT-")]
print(f"Ribosomal genes in HVGs: {len(ribo_in_hvg)}")
print(f"MT genes in HVGs:        {len(mt_in_hvg)}")
print("→ DEG will be run on HVGs only, so these are already excluded.\n")

# Run DEG on HVGs only (RPL/RPS/MT excluded during preprocessing)
adata_deg = adata[:, adata.var["hvg"]].copy()
adata_deg.X = adata_deg.layers["log1p_norm"]

sc.tl.rank_genes_groups(
    adata_deg,
    groupby="leiden_scVI",
    method="wilcoxon",
    key_added="rank_genes_leiden",
    pts=True,
)
adata.uns["rank_genes_leiden"] = adata_deg.uns["rank_genes_leiden"]
del adata_deg

markers = sc.get.rank_genes_groups_df(
    adata, group=None, key="rank_genes_leiden", pval_cutoff=0.05, log2fc_min=0.5
)

cols = ["group", "names", "logfoldchanges", "pvals_adj"]
if "pts" in markers.columns:
    cols.append("pts")

top5 = markers.groupby("group").head(5)[cols]
rename = {"group": "cluster", "names": "gene", "logfoldchanges": "log2FC",
          "pvals_adj": "adj_pval", "pts": "pct_expressing"}
top5 = top5.rename(columns=rename)
print("Top 5 DEGs per cluster:")
display(top5)
top5.to_csv(os.path.join(OUTPUT_DIR, "deg_top5_per_cluster.csv"), index=False)

sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby="leiden_scVI",
    key="rank_genes_leiden",
    n_genes=3,
    save="_deg_dotplot.png",
)

adata.X = adata.layers["counts"]

## Step 6 — scib benchmarking — scVI vs unintegrated PCA

Compares scVI (batch-corrected) against unintegrated PCA on two axes:
- **Batch correction** (iLISI): are the 14 libraries well mixed?
- **Biological conservation** (cLISI, NMI): are cell types still separated?

## Step 6 — Save clean integrated object

In [ ]:
try:
    from scib_metrics.benchmark import Benchmarker

    bio_label = next(
        (c for c in ["cell_type_Scanorama", "cell_type", "celltype"] if c in adata.obs.columns),
        "leiden_scVI"
    )
    print(f"Using '{bio_label}' as biological label")

    bm = Benchmarker(
        adata,
        batch_key=SAMPLE_COL,
        label_key=bio_label,
        embedding_obsm_keys=["X_pca", "X_scVI"],
        n_jobs=-1,
    )
    bm.benchmark()
    results = bm.get_results(min_max_scale=False)

    print("Raw results shape:", results.shape)
    print("Raw results:")
    display(results)

    # Build a clean metric x embedding table
    numeric_cols = [c for c in results.columns if c != "Metric Type"]
    table = results[numeric_cols].copy()
    table.index.name = "Metric"
    table = table.round(3)

    print("\nscib benchmarking — X_pca vs X_scVI:")
    display(table)
    table.to_csv(os.path.join(OUTPUT_DIR, "scib_results.csv"))

except ImportError:
    print("scib-metrics not installed. Run: pip install scib-metrics")

## Save scvi_leiden + scvi_doublet to shared object

In [ ]:
import numpy as np
import pandas as pd

SHARED_PATH  = "/vol/disk/ubuntu/master_practicum_cytokines/data/data_for_practicum_preprocessed_doublet_clusters.h5ad"
METHODS_PATH = "/vol/disk/ubuntu/master_practicum_cytokines/data/data_for_practicum_integration_methods.h5ad"

# ── Save scvi_leiden + scvi_doublet to doublet clusters object ───────────────
adata_shared = sc.read_h5ad(SHARED_PATH)
adata_shared.obs["scvi_leiden"] = adata.obs["leiden_scVI"].reindex(adata_shared.obs_names)
adata_shared.obs["scvi_doublet"] = adata.obs["leiden_scVI"].reindex(adata_shared.obs_names).isin(["1", "16"])
print(f"Doublet cells (clusters 1+16): {adata_shared.obs['scvi_doublet'].sum():,}")
adata_shared.write_h5ad(SHARED_PATH)
print(f"Saved leiden + doublet flags → {SHARED_PATH}")

# ── Save X_scVI + X_umap_scVI to integration methods object ─────────────────
adata_methods = sc.read_h5ad(METHODS_PATH)

# Align cell order: index into adata arrays using methods object cell IDs
cell_idx = pd.Index(adata.obs_names).get_indexer(adata_methods.obs_names)
adata_methods.obsm["X_scVI"]      = adata.obsm["X_scVI"][cell_idx]
adata_methods.obsm["X_umap_scVI"] = adata.obsm["X_umap_scVI"][cell_idx]

adata_methods.write_h5ad(METHODS_PATH)
print(f"Saved X_scVI + X_umap_scVI → {METHODS_PATH}")